# RealMLP Model - PharmShed Super Dataset

**Model:** RealMLP-TD (pytabkit)
**Task:** Multi-class classification — predict which of 218 classes a person is prescribed (217 drugs + "no prescriptions")
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)
**Split strategy:** StratifiedGroupKFold (5-fold CV, shuffle=True, random_state=42), grouped by Person_ID to prevent data leakage
**Missing values:** Prescription NaNs for "no prescription" rows filled with -1 sentinel; Age, Strength, and Day_Supply imputed within each fold using hierarchical median imputation
**Preprocessing:** Categorical columns set to `category` dtype — RealMLP handles encoding and scaling internally. No manual StandardScaler or OneHotEncoder needed.
**Metrics:** Accuracy, Cohen's Kappa, MCC, macro/micro averaged Precision, Recall, F1, F2
**Output:** Saves `realmlp_super_proba_2022.csv` — probability vector over 218 classes per observation, used as input to the ensemble model

## Key Design Decisions

**Why RealMLP-TD?**
RealMLP-TD uses tuned default hyperparameters designed to perform well on tabular data without
manual hyperparameter search. It is a neural network architecture specifically optimized for
tabular classification, making it a strong complement to tree-based models (XGBoost) and
distance-based models (KNN, SVM) in the ensemble.

**Why -1 for "no prescription" NaNs?**
The 97,497 "no prescriptions" rows have NaN for Quantity, Form, Strength, and Day_Supply because
those fields are structurally absent — not unknown. Filling with -1 preserves this as a meaningful
signal distinct from real prescription values. RealMLP scales features internally so -1 maps
to a distinct region away from real values after normalization.

**Why impute Age, Strength, and Day_Supply inside each fold?**
Age, Strength, and Day_Supply have real missing values (not structural absences) that require
imputation. Imputation is applied within each fold — fit on train, applied to val — to prevent
data leakage. Age uses a hierarchical median strategy (household → income/insurance group → year → global).
Strength and Day_Supply use median imputation grouped by year and drug.

**Why category dtype for categorical columns?**
RealMLP detects categorical columns by their dtype. Setting categorical columns to `category`
dtype before fitting tells RealMLP to apply its built-in categorical encoding internally.
Do not one-hot encode manually — this would duplicate encoding and inflate feature dimensionality.

**Why no RobustScaler or StandardScaler?**
RealMLP-TD handles numeric feature scaling internally as part of its preprocessing pipeline.
Applying any external scaler before passing data to RealMLP would double-scale the features
and corrupt the internal preprocessing.

**Note on epochs:**
`n_epochs=30` is resource-constrained due to 16GB RAM limit on the training machine.
More epochs would likely improve recall for rare drug classes but were not feasible.
This limitation is noted in the paper.

In [ ]:
# Install required libraries.
# pytabkit provides RealMLP-TD, permetrics provides macro/micro evaluation metrics.
!pip install pytabkit permetrics

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
import joblib
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from pytabkit import RealMLP_TD_Classifier
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
import sklearn
print('pandas version:      ', pd.__version__)
print('numpy version:       ', np.__version__)
print('scikit-learn version:', sklearn.__version__)
print('torch version:       ', torch.__version__)

# Machine-generic device selection.
# Automatically picks the best available device — no manual changes needed.
# Priority: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'\nDevice selected: {DEVICE}')
print('(cuda = NVIDIA GPU, mps = Apple Silicon, cpu = fallback)')

In [ ]:
# Load the super integrated dataset (2014-2021).
# The super dataset contains demographics + prescription features + Person_ID.

DATA_DIR = './'  # change this to your data path if needed
SUPER_DATA = f'{DATA_DIR}super_integrated_data.csv'

super_df = pd.read_csv(
    SUPER_DATA,
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

Super dataset shape: (905728, 11)
Metadata shape: (905728, 5)

Super dataset columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']

Missing values:
Observation_ID            0
Drug                      0
Age                       0
Sex                       0
Family_income             0
Insurance_coverage        0
Race_ethnicity            0
Quantity              97497
Form                  97497
Strength              97497
Day_Supply            97497
dtype: int64


In [ ]:
# Verify Person_ID is present in the dataset.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
assert 'Person_ID' in super_df.columns, \
    "ERROR: Person_ID not found in dataset — check super_integrated_data.csv."
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs — check super_integrated_data.csv."

print('Person_ID verified.')
print('Unique persons:', super_df['Person_ID'].nunique())

Columns after join: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply', 'Person_ID']
Shape after join: (905728, 12)
Missing Person_IDs: 0


In [ ]:
# Prescription NaN filling for "no prescription" rows only.
# Rows where a person had no prescription have NaN for all prescription features.
# These are filled with -1 as a sentinel value to distinguish them from
# real missing values that will be imputed inside each fold.
numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
mask = super_df['Drug'] == 'no prescriptions'

# Form is categorical string column (values like TABS, ORAL, CAPS).
# Fill with string '-1' directly — avoids mixed int/str type issue.
# RealMLP OrdinalEncoder requires uniform types in categorical columns.
super_df.loc[mask, 'Form'] = super_df.loc[mask, 'Form'].fillna('-1')

print('Missing values after fill:')
print(super_df[['Quantity', 'Form', 'Strength', 'Day_Supply']].isnull().sum())
print('Form dtype:', super_df['Form'].dtype)
print('Form sample values:', super_df['Form'].unique()[:5].tolist())

#assert super_df[numeric_prescription_cols + ['Form']].isnull().sum().sum() == 0, \
 #   "ERROR: Prescription NaNs remain after fill — check fillna logic before proceeding."

#print('\nAll prescription NaNs filled successfully.')

Missing values after filling with -1:
Observation_ID        0
Drug                  0
Age                   0
Sex                   0
Family_income         0
Insurance_coverage    0
Race_ethnicity        0
Quantity              0
Form                  0
Strength              0
Day_Supply            0
Person_ID             0
dtype: int64


In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Copied from Vanessa's TabICL script for consistency across all models.

def impute_age(df):
    df = df.copy()
    df['income_bracket'] = (
        df.groupby('Year')['Family_income']
        .transform(lambda x: pd.qcut(x, 4, labels=False, duplicates='drop') + 1)
    )
    hh_meds    = df.groupby(['Year', 'Household_ID'])['Age'].transform('median')
    grp_meds   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].transform('median')
    yr_meds    = df.groupby('Year')['Age'].transform('median')
    global_med = df['Age'].median()
    df['Age']  = df['Age'].fillna(hh_meds)
    df['Age']  = df['Age'].fillna(grp_meds)
    df['Age']  = df['Age'].fillna(yr_meds)
    df['Age']  = df['Age'].fillna(global_med)
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def impute_strength(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Strength'].transform('median')
    # 2. Median by Drug (across all years)
    drug_meds = df.groupby('Drug')['Strength'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Strength'].median()

    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(yr_drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(drug_meds)
    df.loc[mask, 'Strength'] = df.loc[mask, 'Strength'].fillna(global_med)
    return df

def impute_day_supply(df):
    df = df.copy()
    mask = df['Drug'] != 'no prescriptions'
    # 1. Median by Year and Drug
    yr_drug_meds = df.groupby(['Year', 'Drug'])['Day_Supply'].transform('median')
    # 2. Median by Drug
    drug_meds = df.groupby('Drug')['Day_Supply'].transform('median')
    # 3. Global Median
    global_med   = df.loc[mask, 'Day_Supply'].median()

    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(yr_drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(drug_meds)
    df.loc[mask, 'Day_Supply'] = df.loc[mask, 'Day_Supply'].fillna(global_med)
    return df

In [ ]:
# Hard stop if drug count is not 217 (216 drugs + "no prescriptions").
assert super_df['Drug'].nunique() == 217, \
    f"ERROR: Expected 217 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 217 (216 drugs + no prescriptions).')

Unique persons: 126967
Unique drugs: 217
No prescription rows: 97497
Actual drug rows: 808231

Top 10 most prescribed drugs:
Drug
no prescriptions    97497
atorvastatin        38557
lisinopril          35851
metformin           33777
amlodipine          28135
metoprolol          25001
albuterol           23188
omeprazole          22770
losartan            18670
gabapentin          18337
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
sulfamethoxazole    72
trimethoprim        72
gentamicin          64
piroxicam           61
ivermectin          29
Name: count, dtype: int64


In [ ]:
# Define feature columns, categorical columns, and target.
# Form is categorical because it contains drug form codes (TABS, CAPS, ORAL, etc.).
# Quantity, Strength, Day_Supply are numeric — RealMLP scales these internally.
# RealMLP handles all encoding and scaling internally — do not preprocess further.
feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage',
                    'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_cols     = ['Age', 'Family_income', 'Quantity', 'Strength', 'Day_Supply']
target_col       = 'Drug'

# Fit LabelEncoder once on the full dataset before the CV loop.
# Fitting inside the loop would produce different integer mappings per fold,
# making fold results incomparable and breaking per-drug recall aggregation.
le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes:', len(le.classes_))
print('Feature cols:', feature_cols)
print('Categorical cols:', categorical_cols)
print('\nSample drug to integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'realmlp_super_label_encoder.joblib')
print('\nLabelEncoder saved to realmlp_super_label_encoder.joblib')

Unique classes: 217
Feature cols: ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
Categorical cols: ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']

Sample drug to integer mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4


In [ ]:
# 5-fold stratified group cross-validation.
#
# StratifiedGroupKFold ensures:
#   1. Each fold has similar drug class distribution (stratified)
#   2. All records for the same person stay in the same fold (grouped)
#      preventing the model from seeing the same person in both train and val
#
# For each fold:
#   1. Split by Person_ID groups and drug label stratification
#   2. Impute Age, Strength, Day_Supply within each fold (fit on train, apply to val)
#   3. Set categorical dtypes so RealMLP uses its built-in categorical encoding
#   4. Train RealMLP-TD on train fold
#   5. Predict on val fold
#   6. Compute and store all metrics

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    train_fold = super_df.iloc[train_idx].copy()
    val_fold   = super_df.iloc[val_idx].copy()

    # Impute Age, Strength, Day_Supply within each fold.
    # Imputation is fit on train and applied to val to prevent data leakage.
    train_fold = impute_age(train_fold)
    val_fold   = impute_age(val_fold)

    train_fold = impute_strength(train_fold)
    val_fold   = impute_strength(val_fold)

    train_fold = impute_day_supply(train_fold)
    val_fold   = impute_day_supply(val_fold)

    X_train_fold = train_fold[feature_cols].copy()
    X_val_fold   = val_fold[feature_cols].copy()
    y_train_fold = train_fold['Drug_encoded'].values
    y_val_fold   = val_fold['Drug_encoded'].values

    # set sentinel value for "no prescriptions" rows
    X_train_fold.fillna(-1, inplace=True)
    X_val_fold.fillna(-1, inplace=True)

    # Set categorical dtypes so RealMLP applies its built-in categorical encoding.
    # Do not one-hot encode manually — RealMLP handles this internally.
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs in train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Train RealMLP-TD.
    # TD version uses tuned default hyperparameters — no manual search needed.
    # DEVICE is set automatically based on available hardware.
    model = RealMLP_TD_Classifier(
        device=DEVICE,
        random_state=42,
        n_epochs=30,
    )
    model.fit(X_train_fold, y_train_fold)
    print(f'Fold {fold_num} training complete.')

    y_pred_fold = model.predict(X_val_fold)

    # Overall metrics.
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    # Macro and micro averaged metrics via permetrics.
    # Macro: average per class equally — treats rare and common drugs equally.
    # Micro: aggregate all counts — dominated by the most common drugs.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)

    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')

    # F2 score weights recall twice as much as precision.
    # Higher recall is more important here because missing a drug class
    # means underestimating its wastewater load.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })

    # Per-drug recall for this fold.
    # Used for ensemble model selection — the ensemble picks the best model per drug.
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  Cohen Kappa:   {kappa:.4f}')
    print(f'  MCC:           {mcc:.4f}')
    print(f'  Macro Recall:  {macro_recall:.4f}')
    print(f'  Micro Recall:  {micro_recall:.4f}')
    print(f'  Macro F2:      {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)


FOLD 1/5
Train size: 724,582 | Val size: 181,146
Unique drugs in train: 217 | val: 217


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

In [ ]:
# Summarize cross-validation results across all 5 folds.
# Report mean and standard deviation for each metric.
# Standard deviation shows how stable the model is across different data splits.
results_df = pd.DataFrame(fold_results)

# Hard stop if not all 5 folds completed.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('CV RESULTS — MEAN +/- STD ACROSS 5 FOLDS')
print('='*55)
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:<25}: {mean:.4f} +/- {std:.4f}')

results_df.to_csv('realmlp_super_cv_results.csv', index=False)
print('\nCV results saved to realmlp_super_cv_results.csv')
print('All 5 folds confirmed complete.')

In [ ]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection.
# The ensemble picks whichever base model has the highest recall for each drug.
per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_RealMLP_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_RealMLP_Super', ascending=False)

print('Top 20 drugs by mean recall:')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall:')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does RealMLP recall at all?
# A drug is considered recalled if mean recall > 0 across folds.
# This number goes directly into the paper for model comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_RealMLP_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0:    {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_RealMLP_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_RealMLP_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('realmlp_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to realmlp_super_per_drug_recall.csv')

In [ ]:
# Train final model on the full training dataset (2014-2021).
# After CV gives confidence in model performance, retrain on all available data
# to maximize signal before evaluating on the held-out 2022 validation set.
#
# Note: RealMLP (pytabkit) does not support joblib.dump() or standard pickle saving.
# If the kernel dies after this cell, the model must be retrained from scratch.
# Ensure this cell and the next two cells (2022 validation + save outputs)
# are run in a single uninterrupted session.

# Impute Age, Strength, Day_Supply on full dataset before final training.
data_final = impute_age(super_df.copy())
data_final = impute_strength(data_final)
data_final = impute_day_supply(data_final)

# fill sentinel value for "no prescriptions" numeric columns
data_final.fillna(-1, inplace=True)

X_final = data_final[feature_cols].copy()
y_final = data_final['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

# Shuffle to separate refill records after imputation.
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:', len(le.classes_))
print(f'Device: {DEVICE}')
print('\nTraining final RealMLP model...')

final_model = RealMLP_TD_Classifier(
    device=DEVICE,
    random_state=42,
    n_epochs=30,
)
final_model.fit(X_final, y_final)
print('Final model training complete.')

# save model
joblib.dump(final_model, 'realmlp_super_final_model.joblib')

In [ ]:
# Load 2022 super dataset (held-out internal validation set).
# Must use super_data_2022.csv — not data_2022.csv —
# because the model was trained with prescription features.
#
# Preprocessing must mirror training exactly:
#   - Prescription NaNs filled with -1 for "no prescription" rows
#   - Age, Strength, Day_Supply imputed using same functions as training
#   - Categorical columns set to category dtype (same as training)
#   - No preprocessor.transform() needed — RealMLP handles scaling internally.

data_2022 = pd.read_csv(
    f'{DATA_DIR}super_data_2022.csv',
    sep=None,
    engine='python',
    encoding='utf-8-sig'
)

if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

# Same sentinel filling as training — must be identical preprocessing
numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
mask = data_2022['Drug'] == 'no prescriptions'
data_2022.loc[mask, 'Form'] = data_2022.loc[mask, 'Form'].fillna('-1')

print('2022 data shape:', data_2022.shape)

# Drop any drugs in 2022 not seen during training
unseen = set(data_2022['Drug'].unique()) - set(le.classes_)
print(f'Unseen drugs in 2022 (will be dropped): {unseen}')
data_2022 = data_2022[data_2022['Drug'].isin(le.classes_)].reset_index(drop=True)
print(f'2022 rows after filtering: {len(data_2022):,}')

# Impute Age, Strength, Day_Supply — must mirror training preprocessing.
data_2022 = impute_age(data_2022)
data_2022 = impute_strength(data_2022)
data_2022 = impute_day_supply(data_2022)

# fill sentinel value for "no prescriptions" numeric columns
data_2022.fillna(-1, inplace=True)

X_2022 = data_2022[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022['Drug'])

# Shuffle to separate refill records.
X_2022, y_2022_encoded = shuffle(X_2022, y_2022_encoded, random_state=42)
X_2022 = X_2022.reset_index(drop=True)

y_pred_2022 = final_model.predict(X_2022)

# Compute validation metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

evaluator_2022    = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = evaluator_2022.precision_score(average='macro')
micro_prec_2022   = evaluator_2022.precision_score(average='micro')
macro_recall_2022 = evaluator_2022.recall_score(average='macro')
micro_recall_2022 = evaluator_2022.recall_score(average='micro')
macro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='micro')

print('\nINTERNAL VALIDATION — MEPS 2022 Results')
print(f'Accuracy:          {acc_2022:.4f}')
print(f'Cohen Kappa:       {kappa_2022:.4f}')
print(f'MCC:               {mcc_2022:.4f}')
print(f'Macro Precision:   {macro_prec_2022:.4f}')
print(f'Micro Precision:   {micro_prec_2022:.4f}')
print(f'Macro Recall:      {macro_recall_2022:.4f}')
print(f'Micro Recall:      {micro_recall_2022:.4f}')
print(f'Macro F2:          {macro_f2_2022:.4f}')
print(f'Micro F2:          {micro_f2_2022:.4f}')

# Save probability outputs for ensemble model.
# Shape: (n_observations, 218) — one row per observation, one column per drug class.
# Columns ordered by le.classes_ — identical mapping across all base models.
# RealMLP_TD_Classifier supports predict_proba() natively via pytabkit.
proba_2022 = final_model.predict_proba(X_2022)
proba_df   = pd.DataFrame(proba_2022, columns=le.classes_)
proba_df.insert(0, 'Observation_ID', data_2022['Observation_ID'].reset_index(drop=True).values)
proba_df.to_csv('realmlp_super_proba_2022.csv', index=False)
print(f'\nProbability output saved to realmlp_super_proba_2022.csv')
print(f'Shape: {proba_df.shape} — {len(data_2022):,} observations x 218 drugs + Observation_ID')

In [ ]:
# Save per-drug metrics and overall validation summary for 2022.
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

drug_recall_2022 = pd.DataFrame([
    {
        'Drug': drug,
        'Recall_2022': report_2022[drug]['recall'],
        'Precision_2022': report_2022[drug]['precision'],
        'F1_2022': report_2022[drug]['f1-score'],
        'Support_2022': report_2022[drug]['support']
    }
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print('Top 15 drugs by recall on 2022 data:')
print(drug_recall_2022.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall on 2022 data:')
print(drug_recall_2022.tail(15).to_string(index=False))

# Summary stats — mirrors Cell 10 but on held-out 2022 data instead of CV.
# Confirms whether CV recall numbers generalise to unseen data.
drugs_recalled_2022 = (drug_recall_2022['Recall_2022'] > 0).sum()
print(f'\nDrugs recalled on 2022 data (recall > 0): {drugs_recalled_2022} / {len(drug_recall_2022)}')
print(f'Drugs with recall >= 0.1: {(drug_recall_2022["Recall_2022"] >= 0.1).sum()}')
print(f'Drugs with recall >= 0.5: {(drug_recall_2022["Recall_2022"] >= 0.5).sum()}')

drug_recall_2022.to_csv('realmlp_super_2022_per_drug_metrics.csv', index=False)
print('\nPer-drug 2022 metrics saved to realmlp_super_2022_per_drug_metrics.csv')

validation_summary = pd.DataFrame([{
    'model': 'RealMLP_Super',
    'dataset': 'MEPS_2022_internal_validation',
    'accuracy': acc_2022,
    'cohen_kappa': kappa_2022,
    'mcc': mcc_2022,
    'macro_precision': macro_prec_2022,
    'micro_precision': micro_prec_2022,
    'macro_recall': macro_recall_2022,
    'micro_recall': micro_recall_2022,
    'macro_f2': macro_f2_2022,
    'micro_f2': micro_f2_2022,
}])
validation_summary.to_csv('realmlp_super_validation_summary.csv', index=False)
print('Validation summary saved to realmlp_super_validation_summary.csv')

# Final output file summary — confirms everything saved correctly.
print('\n' + '='*55)
print('ALL OUTPUTS SAVED')
print('='*55)
print('  realmlp_super_cv_results.csv')
print('  realmlp_super_per_drug_recall.csv')
print('  realmlp_super_2022_per_drug_metrics.csv')
print('  realmlp_super_validation_summary.csv')
print('  realmlp_super_proba_2022.csv           <- ensemble input')
print('  realmlp_super_label_encoder.joblib')
print('  NOTE: RealMLP model cannot be saved — retrain from Cell 11 if needed.')

## Output Files

| File | Contents |
|------|----------|
| `realmlp_super_cv_results.csv` | Mean +/- std for all metrics across 5 CV folds |
| `realmlp_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble selection) |
| `realmlp_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 internal validation |
| `realmlp_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |
| `realmlp_super_proba_2022.csv` | Probability vector over 218 classes per observation — input to ensemble model |
| `realmlp_super_label_encoder.joblib` | Saved LabelEncoder for drug class mapping |

**Next step:** Pass `realmlp_super_proba_2022.csv` to the ensemble model (`ensemble_pharmshed.ipynb`) along with probability outputs from all other base models for soft voting ensemble construction.